# 🏛️ Chapter 00: Enterprise Spatial Environment & PostgreSQL Configuration
## PostGIS for Urban Spatial Analytics (Ghana Edition)

### 🎯 The Mission: Why We Are Doing This
Spatial geometries and raster pixels require significantly more memory for sorting, hash joins, and $R$-Tree index construction than traditional text/integer tables. Before executing spatial queries across Greater Accra, we must configure PostgreSQL for spatial workloads, enable core extensions (`postgis`, `postgis_raster`, `pgrouting`), and establish an analytical project schema (`urban_ghana`).

---

### 🔑 Spatial DBA Keywords Decoded

| Keyword / Setting | Plain-English Explanation | Production Importance |
| :--- | :--- | :--- |
| **`postgis`** | The core spatial C-library extension adding geometry data types and spatial functions to PostgreSQL. | Foundation for all vector operations. |
| **`postgis_raster`** | The extension enabling in-database storage and Map Algebra for satellite & elevation grids. | Replaces external raster GIS software. |
| **`pgrouting`** | The graph network routing extension providing shortest-path and isochrone algorithms. | Transforms street centerlines into navigable transit graphs. |
| **`work_mem`** | Dedicated memory allocated for *each* spatial sort, hash join, or aggregation operation. | Prevents queries from spilling onto slow disk drives. |
| **`maintenance_work_mem`** | Memory allocated for index construction (`GIST`). | Accelerates spatial index creation by up to $10\times$. |

---

### ⚠️ The Professor's Pitfall & DBA Warning
> **The Out-of-Memory Trap:** Never set `work_mem` excessively high on multi-user servers (e.g. 4GB). `work_mem` is allocated *per operation per user*. A query with 4 spatial joins across 20 concurrent connections can consume $4 \times 20 \times \text{work_mem}$. Setting `work_mem = '128MB'` is the ideal sweet spot for spatial analysis.

In [ ]:
# Connect to the Database and Initialize Extensions
from config_db import execute_sql, query_df

sql_init = """
CREATE SCHEMA IF NOT EXISTS urban_ghana;
SET search_path TO urban_ghana, public;

CREATE EXTENSION IF NOT EXISTS postgis;
CREATE EXTENSION IF NOT EXISTS postgis_raster;
CREATE EXTENSION IF NOT EXISTS pgrouting;

SET work_mem = '128MB';
SET maintenance_work_mem = '512MB';
"""

execute_sql(sql_init)
print("✅ Database extensions and memory parameters initialized.")

In [ ]:
# Verify Extension Versions
df_version = query_df("""
SELECT 
    postgis_lib_version()  AS postgis_version,
    postgis_geos_version() AS geos_engine,
    postgis_proj_version() AS proj_datum_engine,
    pgr_version()          AS pgrouting_version;
""")

df_version